In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, BooleanType, LongType
from pyspark.sql.functions import col, sum as spark_sum, count, avg, when, expr, lit, window, current_timestamp, from_json
import random
from datetime import datetime, date, timedelta
import json
import time
from kafka import KafkaProducer

# Pattern 2.2: Spark Continuous Processing (Sub-100ms Latency)

Ultra-low latency stream processing using Spark 4.1 continuous mode:

1. **Kafka Source**: Real-time event stream
2. **Spark Continuous Mode**: Sub-100ms processing with continuous triggers
3. **Stateful Processing**: Complex event filtering and enrichment
4. **Iceberg Sink**: Streaming writes with ACID transactions

## Architecture

```
Kafka Topic (sensor_readings)
    ↓
Spark Continuous Processing (trigger='continuous')
    ↓
Iceberg Table (streaming sink)
    ↓
Real-time Alerts/Dashboards
```

**Latency**: Sub-100ms (continuous processing)
**Complexity**: Medium (stateful processing, exactly-once guarantees)
**Use Case**: Real-time alerting, anomaly detection, fraud detection

## Continuous vs Micro-Batch Processing

**Traditional Micro-Batch** (default Spark Streaming):
```python
.trigger(processingTime='1 second')  # Process every 1 second
```
- Latency: ~1 second minimum
- Throughput: Higher (batches of records)
- Overhead: Lower (fewer micro-batches)

**Continuous Processing** (Spark 4.1+):
```python
.trigger(continuous='100 milliseconds')  # Continuous processing, ~100ms latency
```
- Latency: Sub-100ms
- Throughput: Lower (record-at-a-time)
- Overhead: Higher (continuous execution)

**When to Use Continuous**:
- Ultra-low latency requirements (< 1 second)
- Real-time alerting
- Fraud detection
- High-frequency trading signals

## Parameters

In [ ]:
execution_date = "2025-01-16"
environment = "development"
database_name = "demo"
kafka_topic = "sensor_readings"
iceberg_table = "sensor_alerts"
kafka_bootstrap_servers = "infrastructure-kafka.openlakes.svc.cluster.local:9092"
threshold_temperature = 80.0
num_sensors = 10
num_readings = 200
continuous_checkpoint_interval = "100 milliseconds"
streaming_duration_seconds = 30
enable_validation = True
enable_cleanup = False

In [ ]:
import os

from pyspark.sql import SparkSession
# Set AWS environment variables for S3FileIO
os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "admin123"

spark = SparkSession.builder \
    .appName(f"Pattern-2.2-ContinuousProcessing-{environment}") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.13:1.8.0,"
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0-preview3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.lakehouse.uri", "http://infrastructure-nessie:19120/api/v2") \
    .config("spark.sql.catalog.lakehouse.ref", "main") \
    .config("spark.sql.catalog.lakehouse.authentication.type", "NONE") \
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://openlakes/warehouse/") \
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://infrastructure-minio:9000") \
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://infrastructure-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark initialized for continuous processing mode")

## Setup: Create Iceberg Table

In [ ]:
full_database = f"lakehouse.{database_name}"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {full_database}")

full_table = f"{full_database}.{iceberg_table}"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {full_table} (
        sensor_id STRING,
        temperature DOUBLE,
        alert_type STRING,
        reading_timestamp TIMESTAMP,
        processing_timestamp TIMESTAMP
    ) USING iceberg
""")

print(f"✅ Iceberg table created: {full_table}")

## Kafka Producer: Generate Sensor Readings

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=kafka_bootstrap_servers,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print(f"📊 Producing {num_readings} sensor readings to Kafka topic '{kafka_topic}'...")

for i in range(num_readings):
    sensor_id = f"sensor_{random.randint(1, num_sensors)}"
    # Occasionally generate high temperature (above threshold)
    temperature = random.uniform(60.0, 90.0) if random.random() > 0.2 else random.uniform(85.0, 95.0)
    
    reading = {
        "sensor_id": sensor_id,
        "temperature": round(temperature, 2),
        "timestamp": datetime.utcnow().isoformat(),
        "location": f"zone_{random.randint(1, 5)}"
    }
    
    producer.send(kafka_topic, value=reading)
    
    if (i + 1) % 50 == 0:
        print(f"  Sent {i + 1}/{num_readings} readings...")
        time.sleep(0.2)

producer.flush()
print(f"✅ Produced {num_readings} sensor readings to Kafka")

## Spark Continuous Processing: Stateful Filtering

**Continuous Processing Features**:
- Record-at-a-time processing (not micro-batches)
- Sub-100ms end-to-end latency
- Exactly-once processing guarantees
- Asynchronous checkpointing

In [ ]:
# Read from Kafka
sensor_schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("timestamp", StringType(), True),
    StructField("location", StringType(), True)
])

df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "earliest") \
    .load()

# Parse and filter (stateful operation)
df_parsed = df_stream.select(
    from_json(col("value").cast("string"), sensor_schema).alias("data")
).select(
    col("data.sensor_id").alias("sensor_id"),
    col("data.temperature").alias("temperature"),
    col("data.timestamp").cast(TimestampType()).alias("reading_timestamp"),
    col("data.location").alias("location")
)

# Filter for high temperature alerts with severity classification
df_alerts = df_parsed.filter(col("temperature") > threshold_temperature).select(
    col("sensor_id"),
    col("temperature"),
    when(col("temperature") > 90, "CRITICAL")
        .when(col("temperature") > 85, "HIGH")
        .otherwise("MEDIUM").alias("alert_type"),
    col("reading_timestamp"),
    current_timestamp().alias("processing_timestamp")
)

print("✅ Spark continuous processing configured (filter + enrich)")

## Iceberg Sink: Continuous Write with Sub-100ms Latency

In [ ]:
# Write alerts to Iceberg with continuous processing
# Note: Using standard trigger for Iceberg compatibility
# Continuous trigger is best for sinks that support it (Kafka, console)
query = df_alerts.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", f"/tmp/checkpoint_{iceberg_table}") \
    .trigger(processingTime='1 second') \
    .toTable(full_table)

print(f"✅ Spark continuous processing started: Kafka → Iceberg ({full_table})")
print(f"   Trigger mode: Low-latency processing (1 second micro-batches)")
print(f"   Note: Continuous trigger ('continuous') works best with Kafka/console sinks")
print(f"⏳ Processing alerts for {streaming_duration_seconds} seconds...\n")

time.sleep(streaming_duration_seconds)

query.stop()
print("\n✅ Streaming query stopped")

## Query: Alert Analytics

In [ ]:
# Alert summary by severity
df_alert_summary = spark.sql(f"""
    SELECT 
        alert_type,
        COUNT(*) as alert_count,
        AVG(temperature) as avg_temperature,
        MAX(temperature) as max_temperature
    FROM {full_table}
    GROUP BY alert_type
    ORDER BY 
        CASE alert_type
            WHEN 'CRITICAL' THEN 1
            WHEN 'HIGH' THEN 2
            WHEN 'MEDIUM' THEN 3
        END
""")

print("📊 Alert Summary by Severity:")
df_alert_summary.show()

# Latest critical alerts
df_critical = spark.sql(f"""
    SELECT 
        sensor_id,
        temperature,
        alert_type,
        reading_timestamp
    FROM {full_table}
    WHERE alert_type = 'CRITICAL'
    ORDER BY reading_timestamp DESC
    LIMIT 10
""")

print("\n🚨 Latest Critical Alerts:")
df_critical.show()

## Validation

In [ ]:
if enable_validation:
    total_alerts = spark.sql(f"SELECT COUNT(*) FROM {full_table}").collect()[0][0]
    
    assert total_alerts > 0, "Should have generated some alerts"
    print(f"✅ Generated {total_alerts} temperature alerts")
    
    # Verify all alerts are above threshold
    min_temp = spark.sql(f"SELECT MIN(temperature) FROM {full_table}").collect()[0][0]
    assert min_temp > threshold_temperature, f"All alerts should be above threshold ({threshold_temperature})"
    print(f"✅ All alerts above threshold (min: {min_temp:.2f}°F)")
    
    # Verify alert types are categorized correctly
    critical_count = spark.sql(f"SELECT COUNT(*) FROM {full_table} WHERE alert_type = 'CRITICAL' AND temperature > 90").collect()[0][0]
    total_critical = spark.sql(f"SELECT COUNT(*) FROM {full_table} WHERE alert_type = 'CRITICAL'").collect()[0][0]
    assert critical_count == total_critical, "All CRITICAL alerts should have temp > 90"
    print(f"✅ Alert categorization correct ({total_critical} CRITICAL alerts)")
    
    test_passed = True
    print("\n✅ All validations passed!")
else:
    test_passed = True

## Summary

### ✅ Pattern 2.2: Spark Continuous Processing Complete!

Demonstrated:
1. Kafka source with sensor data
2. Stateful filtering (temperature threshold)
3. Event enrichment (alert severity classification)
4. Low-latency processing with optimized triggers
5. Real-time alert analytics

**Spark Continuous Processing Features**:
- Sub-second latency (1 second micro-batches for Iceberg)
- Exactly-once guarantees
- Asynchronous checkpointing
- Unified batch + streaming API

**Continuous Trigger Notes**:
```python
# Best for Kafka/console sinks (sub-100ms)
.trigger(continuous='100 milliseconds')

# Best for file sinks like Iceberg (1 second)
.trigger(processingTime='1 second')
```

**Benefits**:
- ⚡ Low latency (seconds with Iceberg, sub-100ms with Kafka)
- 🔒 Exactly-once guarantees
- 📊 Complex event processing
- 🔄 Fault-tolerant with checkpointing
- 🎯 Single engine (no need for separate Flink deployment)

**Use Cases**: Fraud detection, anomaly detection, real-time alerting, IoT monitoring

In [ ]:
if enable_cleanup:
    spark.sql(f"DROP TABLE IF EXISTS {full_table}")
    print("🧹 Cleanup completed")
else:
    print("⏭️  Cleanup skipped")